In [74]:
# --- IMPORTS ---


import os
import glob
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from scipy.signal import find_peaks

In [99]:
# --- TUNING ROOM ---


class Tuning:

    USE_DUMMY_DATA = False

    TURNING_ANGLE_THRESHOLD = 0.00055    # Sensitivity for detecting a turn, in radians/meter
    MIN_CORNER_DURATION_M = 23.0        # Minimum length a corner must have
    MAX_STRAIGHT_IN_COMPLEX_M = 35.0    # Maximum length of straight between two corners to be considered the same section
    MIN_FINAL_SECTION_LENGTH_M = 400.0

    N_ESTIMATORS = 100     # Nr of trees in the Random Forest model
    RANDOM_STATE = 42       # Seed for reproducibility of model training

    EXPORT_TO_CSV = True
    overwrite = None

In [100]:
# --- HELPER FUNCTIONS ---


# Calculate distance, heading and turning angles

def calculate_base_geometry(df_coords):

    df = df_coords.copy()

    # Calculate the distance between each point and the cumulative distance
    points = df[['x', 'y']].to_numpy()
    deltas = np.diff(points, axis=0)
    step_distances = np.linalg.norm(deltas, axis=1)
    df['Distance'] = np.insert(np.cumsum(step_distances), 0, 0)

    # Calculate the heading angle
    dx = np.gradient(df['x'])
    dy = np.gradient(df['y'])
    df['heading_angle'] = np.arctan2(dy, dx)

    # Calculate turning angle (rate of change of heading angle)
    unwrapped_heading = np.unwrap(df['heading_angle'])
    df['turning_angle'] = np.gradient(unwrapped_heading) / np.gradient(df['Distance'])

    df.fillna(0, inplace=True)
    return df


# Pass 1: identifies all geometric corners and groups them into sections

def find_geometric_corners(df_geom):

    
    # --- 1. find all individual turning events ---
    
    is_turning = df_geom['turning_angle'].abs() > Tuning.TURNING_ANGLE_THRESHOLD

    # Identify every start and end of a continuous turning event
    turn_changes = is_turning.astype(int).diff()
    turn_starts_idx = df_geom.index[turn_changes == 1]
    turn_ends_idx = df_geom.index[turn_changes == -1]

    if len(turn_ends_idx) < len(turn_starts_idx):
        turn_ends_idx = turn_ends_idx.append(pd.Index([len(df_geom)-1]))

    
    # --- 2. Filter and consolidate turns into a list ---
    
    raw_corners = []
    for start_idx, end_idx in zip(turn_starts_idx, turn_ends_idx):
        if start_idx >= end_idx: continue

        start_m = df_geom.loc[start_idx, 'Distance']
        end_m = df_geom.loc[end_idx, 'Distance']

        # Filter out insignificant kinks in the track
        if (end_m - start_m) >= Tuning.MIN_CORNER_DURATION_M:

            corner_slice = df_geom.loc[start_idx:end_idx]
            severity = corner_slice['turning_angle'].abs().sum()

            raw_corners.append({
                'start_idx': start_idx, 'end_idx': end_idx,
                'start_m': start_m, 'end_m': end_m,
                'severity': severity
            })
        
    
    # --- 3. Group raw corners into complexes (chicanes, etc.) ---
    
    if not raw_corners: return []

    corner_complexes = []
    current_complex = raw_corners[0].copy()

    for next_corner in raw_corners[1:]:
        straight_length = next_corner['start_m'] - current_complex['end_m']

        # If the straight is short, merge it into the complex
        if straight_length < Tuning.MAX_STRAIGHT_IN_COMPLEX_M:
            current_complex['end_idx'] = next_corner['end_idx']
            current_complex['end_m'] = next_corner['end_m']
            current_complex['severity'] += next_corner['severity']
        
        else:
            corner_complexes.append(current_complex)
            current_complex = next_corner.copy()
    
    corner_complexes.append(current_complex)


    # --- 4. Calculate final features for each complex ---

    for i, complex_info in enumerate(corner_complexes):
        if i == 0:  # case for first section, which includes start/finish straight
            preceding_straight_length = complex_info['start_m']
        else:
            prev_complex_end_m = corner_complexes[i-1]['end_m']
            preceding_straight_length = complex_info['start_m'] - prev_complex_end_m
        
        complex_info['preceding_straight_length'] = preceding_straight_length

    return corner_complexes


# Matches sections to geometric corners and calculates features

def generate_training_data(training_track_names, coord_base_path, sections_base_path):

    all_corners_data = []
    print("\n--- Generating Training Data from Known Tracks ---")

    for track_name in sorted(list(training_track_names)):
        print(f"   Processing traning track: '{track_name}'...")

        coord_path = os.path.join(coord_base_path, f"{track_name}.csv")
        sections_path = os.path.join(sections_base_path, f"sections_{track_name}.csv")
        df_coords = pd.read_csv(coord_path)
        df_user_sections = pd.read_csv(sections_path)

        # Calculate base geometry and find all geometric corners
        df_geom = calculate_base_geometry(df_coords)
        geometric_corners = find_geometric_corners(df_geom)

        # Match user sections to geometric corners
        for _, user_section in df_user_sections.iterrows():
            user_start_m = user_section['start_m']
            if user_start_m == 0: continue

            # Find the first geometric corner that starts AFTER the user's braking point
            matched_corner = next(
                (gc for gc in geometric_corners if gc['start_m'] > user_start_m),
                None
            )

            if matched_corner:
                braking_distance = matched_corner['start_m'] - user_start_m

                all_corners_data.append({
                    'preceding_straight_length': matched_corner['preceding_straight_length'],
                    'severity': matched_corner['severity'],
                    'braking_distance': braking_distance
                })

    if not all_corners_data:
        return None
    
    return pd.DataFrame(all_corners_data)


# Constructs the final sections from the list of predicted start meters, applying the special start/finish line rules

def assemble_final_sections(predicted_starts_m, total_track_length):

    # Start with a clean, sorted list of unique braking points. DO NOT add 0.0 yet.
    final_starts = sorted(list(set(predicted_starts_m)))
    
    sections = []

    # This logic handles the case where predictions are made.
    if len(final_starts) > 0:
        
        # --- SPECIAL LOGIC FOR SECTION 0 ---
        # The first section starts at 0.0 and ends at the START of the second analytic section.
        # The second analytic section starts at the second predicted braking point (index 1).
        # If there's only one predicted braking point, the first section is the whole lap.
        first_section_end_m = final_starts[1] if len(final_starts) > 1 else total_track_length
        
        sections.append({
            'segment_id': 0,
            'start_m': 0.0,
            'end_m': first_section_end_m
        })

        # --- SUBSEQUENT SECTIONS ---
        # Loop from the second predicted point (index 1) to the end.
        for i in range(1, len(final_starts)):
            start_m = final_starts[i]
            end_m = final_starts[i+1] if i+1 < len(final_starts) else total_track_length

            sections.append({
                'segment_id': i,
                'start_m': start_m,
                'end_m': end_m
            })

    return pd.DataFrame(sections)


In [101]:
# --- MAIN EXECUTION ---


def main():

    # --- 1. Set up paths and identify tracks ---

    data_type = "Dummy" if Tuning.USE_DUMMY_DATA else "Real"
    base_path = os.getcwd()
    coord_base_path = os.path.join(base_path, "Coordinates")
    sections_base_path = os.path.join(base_path, "Sections", data_type)
    print(f"\n--- Using {data_type.upper()} data from: {base_path} ---")

    all_coord_files = glob.glob(os.path.join(coord_base_path, "*.csv"))
    all_track_names = {os.path.basename(f).replace('.csv', '') for f in all_coord_files}
    section_files = [f for f in glob.glob(os.path.join(sections_base_path, "sections_*.csv")) if "target" not in os.path.basename(f)]
    training_track_names = {os.path.basename(f).replace('sections_', '').replace('.csv', '') for f in section_files}
    target_track_names = list(all_track_names - training_track_names)
    
    if not training_track_names:
        print(f"\n[FATAL ERROR] No training tracks found in '{sections_base_path}'. Exiting.")
        return
    if not target_track_names:
        print(f"\n[INFO] No new target tracks to predict on. Exiting.")
        return
    
    print(f"Found {len(training_track_names)} tracks for training: {sorted(list(training_track_names))}")
    print(f"Found {len(target_track_names)} tracks for prediction: {sorted(list(target_track_names))}")


    # --- 2. Generate training data and train the model ---

    master_training_df = generate_training_data(training_track_names, coord_base_path, sections_base_path)

    if master_training_df is None or master_training_df.empty:
        print("\n[FATAL ERROR] Could not generate any training data from the provided tracks. Exiting.")
        return
    
    print(f"\n--- Training Master Regression Model ---")

    features = ['preceding_straight_length', 'severity']
    target = 'braking_distance'

    X_train = master_training_df[features]
    y_train = master_training_df[target]

    # Initialise and train the Random Forest Regressor model
    model = RandomForestRegressor(
        n_estimators=Tuning.N_ESTIMATORS,
        random_state=Tuning.RANDOM_STATE,
        n_jobs=1
    )
    model.fit(X_train, y_train)
    print("Master Regression Model trained successfully.")


    # --- 3. Predict on target tracks ---

    print("\n--- Starting prediction on target tracks ---")
    overwrite_all = Tuning.overwrite

    for target_track in sorted(target_track_names):
        print(f"\n>> Predicting on target track: '{target_track.upper()}' <<")
        target_coord_path = os.path.join(coord_base_path, f"{target_track}.csv")

        try:
            df_target_coords = pd.read_csv(target_coord_path)
        except FileNotFoundError:
            print(f"   [Error] Coordinate file not found: {target_coord_path}. Skipping.")
            continue

        # Pass 1: Find geometric corners on the new track
        df_target_geom = calculate_base_geometry(df_target_coords)
        target_corners = find_geometric_corners(df_target_geom)
        print(f"   Pass 1: Identified {len(target_corners)} geometric corner complexes.")

        if not target_corners:
            print("   [Warning] No corners were detected on this track. Cannot create sections.")
            continue

               

        # Pass 2: Predict braking distance for each corner
        df_predict_features = pd.DataFrame(target_corners)[features]
        predicted_braking_distances = model.predict(df_predict_features)
        print("   Pass 2: Predicted braking distance for each corner.")

        predicted_starts = [
            max(0.0, corner['start_m'] - brake_dist)
            for corner, brake_dist in zip(target_corners, predicted_braking_distances)
        ]

        predicted_starts.sort()
        total_length = df_target_geom['Distance'].iloc[-1]

        if predicted_starts and (total_length-predicted_starts[-1] < Tuning.MIN_FINAL_SECTION_LENGTH_M):
            print(f"   Filtering: Removed last start point {predicted_starts[-1]:.2f}m as it's too close to the finish line.")
            predicted_starts.pop()


        # --- 4. Assemble and export final sections ---

        total_length = df_target_geom['Distance'].iloc[-1]
        
        target_corners_df = pd.DataFrame(target_corners)
        target_corners_df['predicted_braking_distance'] = predicted_braking_distances
        
        final_sections_df = assemble_final_sections(predicted_starts, total_length)
        print(f"   Assembled {len(final_sections_df)} final sections.")

        if Tuning.EXPORT_TO_CSV:
            output_filename = f"sections_target_{target_track}.csv"
            output_path = os.path.join(sections_base_path, output_filename)

            should_write = False
            if not os.path.exists(output_path):
                should_write = True
            elif overwrite_all is True:
                should_write = True
            elif overwrite_all is None:
                choice = input(f"Output file '{output_filename}' exists. Overwrite this and ALL subsequent files? (y/n): ").lower()
                if choice in ['y', 'yes']:
                    overwrite_all = True
                    Tuning.overwrite = True
                    should_write = True
                else:
                    overwrite_all = False
                    Tuning.overwrite = False
                    print(f"   Skipped existing file: {output_path}")
            
            if should_write:
                final_sections_df.to_csv(output_path, index=False)
                print(f"   File Exported: {output_path}")
        
    print("\n--- All processing complete. ---")
    

if __name__ == "__main__":
    main()


--- Using REAL data from: h:\My Drive\Cranfield\11. Individual Thesis\Code ---
Found 6 tracks for training: ['brands_gp', 'donington_gp', 'hungaroring', 'silverstone_gp', 'spa', 'zandvoort']
Found 1 tracks for prediction: ['monza']

--- Generating Training Data from Known Tracks ---
   Processing traning track: 'brands_gp'...
   Processing traning track: 'donington_gp'...
   Processing traning track: 'hungaroring'...
   Processing traning track: 'silverstone_gp'...
   Processing traning track: 'spa'...
   Processing traning track: 'zandvoort'...

--- Training Master Regression Model ---
Master Regression Model trained successfully.

--- Starting prediction on target tracks ---

>> Predicting on target track: 'MONZA' <<
   Pass 1: Identified 8 geometric corner complexes.
   Pass 2: Predicted braking distance for each corner.
   Assembled 8 final sections.
   File Exported: h:\My Drive\Cranfield\11. Individual Thesis\Code\Sections\Real\sections_target_monza.csv

--- All processing compl